![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform On-Demand (Real-Time) Features

In this recipe we define an **on-demand feature** in [**Featureform**](https://docs.featureform.com/) — a feature computed **at request time**, from the live request payload combined with a precomputed feature served from **Redis**.

## Why on-demand features
Some features can't be precomputed because they depend on data that only exists *at the moment of the request* — the amount of the transaction being scored right now, the user's current cart, the time of day. On-demand features let you express that last-mile computation **as a versioned Featureform resource** instead of scattered application code, so the logic that ran in training is the exact logic that runs in production.

## What we'll build
A fraud-style **risk ratio**: `incoming transaction amount ÷ the user's historical average`.
- The **historical average** is a normal feature, precomputed by a SQL transformation and materialized to **Redis**.
- The **incoming amount** is passed in at request time as a parameter.
- An **on-demand feature** combines the two when you call `client.features(...)`.

## The stack — all local, no Spark

- **ClickHouse** — offline store; runs the SQL transformation for the historical average.
- **Redis** — online store; serves that average at low latency.
- **Featureform** coordinator.

> ⚠️ **Needs local Docker; will not run on Colab or in CI.** Requires a running Featureform coordinator (gRPC `localhost:7878`, dashboard `http://localhost` — [install docs](https://docs.featureform.com/deployment/quickstart-docker)) plus the two containers below.

### Start ClickHouse and Redis

In [ ]:
# NBVAL_SKIP
!docker run -d --name clickhouse -p 8123:8123 -p 9000:9000 clickhouse/clickhouse-server:latest
!docker run -d --name redis -p 6379:6379 redis:8

## Environment Setup

### Install Python Dependencies

In [ ]:
%pip install -q featureform redis clickhouse-connect numpy

### Configure connections

In [ ]:
import os

# Featureform coordinator (gRPC)
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

# Address the coordinator container uses to reach the providers.
# Mac/Windows: 'host.docker.internal'. Linux: try '172.17.0.1'.
PROVIDER_HOST = os.getenv("PROVIDER_HOST", "host.docker.internal")

# ClickHouse offline store (default user, no password)
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", PROVIDER_HOST)
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

# Redis online store
REDIS_HOST = os.getenv("REDIS_HOST", PROVIDER_HOST)
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

### Create a sample transactions table in ClickHouse

We load a small transactions table so the average-transaction feature has data to aggregate.

In [ ]:
# NBVAL_SKIP
import clickhouse_connect
import numpy as np

ch = clickhouse_connect.get_client(host="localhost", port=8123,
                                   username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD)
ch.command("DROP TABLE IF EXISTS transactions")
ch.command(
    """
    CREATE TABLE transactions (
        TransactionID String,
        CustomerID String,
        TransactionAmount Float64
    ) ENGINE = MergeTree ORDER BY CustomerID
    """
)
rng = np.random.default_rng(42)
rows = [[f"T{i:05d}", f"C{int(rng.integers(1000, 1050)):04d}", round(float(rng.gamma(2.0, 50.0)), 2)]
        for i in range(500)]
ch.insert("transactions", rows, column_names=["TransactionID", "CustomerID", "TransactionAmount"])
print("rows:", ch.command("SELECT count() FROM transactions"))

## Register providers, source, and the precomputed feature

Standard setup: register ClickHouse + Redis, a SQL transformation for each user's average transaction, and a feature materialized to Redis. This is the value the on-demand feature will build on.

In [ ]:
import featureform as ff

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store with transaction history",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

In [ ]:
transactions = clickhouse.register_table(
    name="transactions", variant="quickstart", table="transactions",
)

@clickhouse.sql_transformation(variant="quickstart")
def average_user_transaction():
    return (
        "SELECT CustomerID AS user_id, avg(TransactionAmount) AS avg_transaction_amt "
        "FROM {{transactions.quickstart}} GROUP BY CustomerID"
    )

@ff.entity
class User:
    avg_transactions = ff.Feature(
        average_user_transaction[["user_id", "avg_transaction_amt"]],
        variant="quickstart",
        type=ff.Float32,
        inference_store=redis,
    )

## Define the on-demand feature

An on-demand feature is a function decorated with `@ff.ondemand_feature`. Its signature is fixed: `(client, params, entities)`.
- `client` — lets the function look up other (precomputed) features, e.g. from Redis.
- `entities` — the entity keys passed at serving time.
- `params` — arbitrary request-time inputs you supply per call.

Here it fetches the user's stored average from Redis and divides the live amount by it. This function is registered and versioned like any feature — but it runs **client-side, at request time**.

In [ ]:
@ff.ondemand_feature(variant="quickstart")
def transaction_risk_ratio(client, params, entities):
    """Live transaction amount relative to the user's historical average."""
    avg = client.features([("avg_transactions", "quickstart")], {"user": entities["user"]})[0]
    incoming_amount = params[0]
    if not avg:
        return 0.0
    return float(incoming_amount) / float(avg)

## Apply

`client.apply()` materializes the average into Redis and registers the on-demand feature definition.

In [ ]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

## Serve it — combine live input with the Redis-served average

Pass the entity and the request-time `params` to `client.features()`. The same call would run behind a live fraud model: a ratio well above 1 means this transaction is large relative to the user's norm.

In [ ]:
# NBVAL_SKIP
user_id = client.dataframe(average_user_transaction)["user_id"].iloc[0]
stored_avg = client.features([("avg_transactions", "quickstart")], {"user": user_id})[0]

for incoming_amount in [stored_avg, stored_avg * 5]:
    ratio = client.features(
        [transaction_risk_ratio],
        {"user": user_id},
        params=[incoming_amount],
    )
    print(f"user {user_id}: amount={incoming_amount:.2f}  avg={stored_avg:.2f}  risk_ratio={ratio}")

### Why this matters

The division logic lives in **one versioned resource**, not duplicated across a training script and a serving service. Train on `transaction_risk_ratio` and you score production traffic with byte-for-byte the same computation — no training-serving skew, even for the real-time part.

## Cleanup

Stop and remove the containers when you're done.

In [ ]:
# NBVAL_SKIP
!docker rm -f clickhouse redis

## Learn more

- [Featureform on-demand features](https://docs.featureform.com/)
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb)